In [1]:
import duckdb
import os

In [2]:
con = duckdb.connect()
con.load_extension("spatial")

In [4]:
n = 0
for i in os.listdir('./output'):
    if i.startswith('n') and i.endswith('.parquet'):
        n0 = int(con.sql(f"SELECT count(*) FROM read_parquet('./output/{i}')").fetchone()[0])
        n += n0
        print(i, n0)
print(f"{' '*(len(i)-5)}total {n}")

n43_w64.parquet 1
n43_w65.parquet 22
n43_w66.parquet 3
n44_w62.parquet 11
n44_w63.parquet 24
n44_w64.parquet 30
n44_w65.parquet 24
n44_w66.parquet 13
n44_w67.parquet 1
n45_w59.parquet 2
n45_w60.parquet 17
n45_w61.parquet 40
n45_w62.parquet 29
n45_w63.parquet 37
n45_w64.parquet 40
n45_w65.parquet 26
n45_w66.parquet 41
n45_w67.parquet 28
n45_w70.parquet 11
n45_w71.parquet 38
n45_w72.parquet 48
n45_w73.parquet 60
n45_w74.parquet 33
n45_w75.parquet 13
n45_w76.parquet 5
n46_w59.parquet 3
n46_w60.parquet 39
n46_w61.parquet 9
n46_w62.parquet 28
n46_w63.parquet 31
n46_w64.parquet 36
n46_w65.parquet 44
n46_w66.parquet 40
n46_w67.parquet 46
n46_w69.parquet 7
n46_w70.parquet 55
n46_w71.parquet 65
n46_w72.parquet 62
n46_w73.parquet 47
n46_w74.parquet 39
n46_w75.parquet 37
n46_w76.parquet 37
n46_w77.parquet 32
n46_w78.parquet 24
n47_w64.parquet 6
n47_w65.parquet 38
n47_w66.parquet 47
n47_w67.parquet 46
n47_w68.parquet 53
n47_w69.parquet 56
n47_w70.parquet 62
n47_w71.parquet 62
n47_w72.parquet 48
n4

In [1]:
fname = 'abdu_cn_east_v2'

In [ ]:
#merge all parquet files in ./output/*.parquet

# # fname = os.path.basename(params['hucsurl']).replace('.parquet', 'wH2Ocalc') #.rsplit('\\',1)[1]
print(fname)
con.execute(f"""
COPY (SELECT * FROM './output/n*.parquet') TO './output/{fname}.parquet' (FORMAT 'parquet');
""")

abdu_cn_east_v2


In [7]:
con.sql(f"SELECT count(*) FROM read_parquet('./output/{fname}.parquet')")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2382 │
└──────────────┘

In [9]:
hucsurl = r"D:\ABDUBounds\ABDU_Canada_Data\parquet\abdu_cn_east_wsheds.parquet"
con.sql(f"SELECT count(*) FROM read_parquet('{hucsurl}')")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2382 │
└──────────────┘

# Aggregate calculations for hucs

In [280]:
cols = con.sql('describe (SELECT * EXCLUDE WATERSHED_CODE_1 FROM abdu_output)').df()['column_name'].tolist()
cols = [i for i in cols if i!='geometry']
cols = [cols[0],cols[2],cols[1],*cols[3:]]
cols

['WATERSHED_CODE',
 'dgr_blk',
 'wshed_ha',
 'dud_lta',
 'demand_lta_kcal',
 'popobj_lta',
 'dud_80th',
 'demand_80th_kcal',
 'popobj_80th',
 'abdu_ltadud',
 'abdu_ltademand',
 'abdu_ltapopobj',
 'abdu_x80dud',
 'abdu_x80demand',
 'abdu_x80popobj',
 'agwt_ltadud',
 'agwt_ltademand',
 'agwt_ltapopobj',
 'agwt_x80dud',
 'agwt_x80demand',
 'agwt_x80popobj',
 'amwi_ltadud',
 'amwi_ltademand',
 'amwi_ltapopobj',
 'amwi_x80dud',
 'amwi_x80demand',
 'amwi_x80popobj',
 'bwte_ltadud',
 'bwte_ltademand',
 'bwte_ltapopobj',
 'bwte_x80dud',
 'bwte_x80demand',
 'bwte_x80popobj',
 'gadw_ltadud',
 'gadw_ltademand',
 'gadw_ltapopobj',
 'gadw_x80dud',
 'gadw_x80demand',
 'gadw_x80popobj',
 'mall_ltadud',
 'mall_ltademand',
 'mall_ltapopobj',
 'mall_x80dud',
 'mall_x80demand',
 'mall_x80popobj',
 'nopi_ltadud',
 'nopi_ltademand',
 'nopi_ltapopobj',
 'nopi_x80dud',
 'nopi_x80demand',
 'nopi_x80popobj',
 'wodu_ltadud',
 'wodu_ltademand',
 'wodu_ltapopobj',
 'wodu_x80dud',
 'wodu_x80demand',
 'wodu_x80popo

In [281]:
hucidfld = cols[0]
hucidfld

'WATERSHED_CODE'

In [282]:
col_sql = f"{cols[0]},string_agg({cols[1]},', ') AS {cols[1]}"
for i in cols[2:]:
    if i.endswith('_pct_ha'):
        col_sql += f",sum(({i}/100)*tothab_ha) AS {i}"
    elif i.endswith('_pct_kcal'):
        col_sql += f",sum(({i}/100)*tothabitat_kcal) AS {i}"
    else:
        col_sql += f",sum({i}) AS {i}"
# ','.join([f"sum(({i}/100)*tothab_ha) AS {i}" for i in cols[2:-1] if i.endswith('pct')])
col_sql

"WATERSHED_CODE,string_agg(dgr_blk,', ') AS dgr_blk,sum(wshed_ha) AS wshed_ha,sum(dud_lta) AS dud_lta,sum(demand_lta_kcal) AS demand_lta_kcal,sum(popobj_lta) AS popobj_lta,sum(dud_80th) AS dud_80th,sum(demand_80th_kcal) AS demand_80th_kcal,sum(popobj_80th) AS popobj_80th,sum(abdu_ltadud) AS abdu_ltadud,sum(abdu_ltademand) AS abdu_ltademand,sum(abdu_ltapopobj) AS abdu_ltapopobj,sum(abdu_x80dud) AS abdu_x80dud,sum(abdu_x80demand) AS abdu_x80demand,sum(abdu_x80popobj) AS abdu_x80popobj,sum(agwt_ltadud) AS agwt_ltadud,sum(agwt_ltademand) AS agwt_ltademand,sum(agwt_ltapopobj) AS agwt_ltapopobj,sum(agwt_x80dud) AS agwt_x80dud,sum(agwt_x80demand) AS agwt_x80demand,sum(agwt_x80popobj) AS agwt_x80popobj,sum(amwi_ltadud) AS amwi_ltadud,sum(amwi_ltademand) AS amwi_ltademand,sum(amwi_ltapopobj) AS amwi_ltapopobj,sum(amwi_x80dud) AS amwi_x80dud,sum(amwi_x80demand) AS amwi_x80demand,sum(amwi_x80popobj) AS amwi_x80popobj,sum(bwte_ltadud) AS bwte_ltadud,sum(bwte_ltademand) AS bwte_ltademand,sum(bwte_l

In [283]:
con.execute(f"""
            CREATE OR REPLACE TABLE abdu_output_agg AS 
            SELECT {col_sql}
            FROM abdu_output
            GROUP BY {cols[0]}
            """)

In [284]:
col_sql = cols[0]
for i in cols[1:]:
    if i.endswith('_pct_ha'):
        col_sql += f",{i} AS {i.replace('_pct','')}"
        col_sql += f",({i}/tothab_ha)*100 AS {i}"
    elif i.endswith('_pct_kcal'):
        col_sql += f",{i} AS {i.replace('_pct','')}"
        col_sql += f",{i}/tothabitat_kcal AS {i}"
    else:
        col_sql += f",{i}"
col_sql


'WATERSHED_CODE,dgr_blk,wshed_ha,dud_lta,demand_lta_kcal,popobj_lta,dud_80th,demand_80th_kcal,popobj_80th,abdu_ltadud,abdu_ltademand,abdu_ltapopobj,abdu_x80dud,abdu_x80demand,abdu_x80popobj,agwt_ltadud,agwt_ltademand,agwt_ltapopobj,agwt_x80dud,agwt_x80demand,agwt_x80popobj,amwi_ltadud,amwi_ltademand,amwi_ltapopobj,amwi_x80dud,amwi_x80demand,amwi_x80popobj,bwte_ltadud,bwte_ltademand,bwte_ltapopobj,bwte_x80dud,bwte_x80demand,bwte_x80popobj,gadw_ltadud,gadw_ltademand,gadw_ltapopobj,gadw_x80dud,gadw_x80demand,gadw_x80popobj,mall_ltadud,mall_ltademand,mall_ltapopobj,mall_x80dud,mall_x80demand,mall_x80popobj,nopi_ltadud,nopi_ltademand,nopi_ltapopobj,nopi_x80dud,nopi_x80demand,nopi_x80popobj,wodu_ltadud,wodu_ltademand,wodu_ltapopobj,wodu_x80dud,wodu_x80demand,wodu_x80popobj,nsho_ltadud,nsho_x80dud,nsho_ltapopobj,nsho_x80popobj,nsho_ltademand,nsho_x80demand,tothabitat_kcal,protected_kcal,protectedhabitat_ha,urbanHa,urbanNrgy,unavailha,surpdef_lta_kcal,surpdef_80th_kcal,nrgprot_lta_kcal,nrgprot

In [285]:
con.execute(f"""
            CREATE OR REPLACE TABLE abdu_output_agg AS 
            SELECT {col_sql}
            FROM abdu_output_agg
            ORDER by {cols[0]}
            """)

In [286]:
con.sql(f"SELECT count(*) FROM abdu_output_agg")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          306 │
└──────────────┘

In [287]:
con.sql(f"SELECT * FROM abdu_output_agg")

┌────────────────┬──────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────

# Calculate weighted mean

In [288]:
cols = [i for i in con.sql('describe abdu_output_agg').df()['column_name'].tolist() if i.endswith('_pct_kcal')]
cols

['DeepwaterFresh_pct_kcal',
 'FreshMarsh_pct_kcal',
 'FreshShallowOpenWater_pct_kcal',
 'FreshwaterWoody_pct_kcal',
 'MudflatSalt_pct_kcal',
 'SaltMarshNonDominant_pct_kcal']

In [289]:
col_sql = ','.join([f"ifnull({i}*({i.replace('_pct','')}/{i.replace('_pct_kcal','_ha')}),0) AS {i.replace('_pct_kcal','_wtmean')}" for i in cols])
col_sql

'ifnull(DeepwaterFresh_pct_kcal*(DeepwaterFresh_kcal/DeepwaterFresh_ha),0) AS DeepwaterFresh_wtmean,ifnull(FreshMarsh_pct_kcal*(FreshMarsh_kcal/FreshMarsh_ha),0) AS FreshMarsh_wtmean,ifnull(FreshShallowOpenWater_pct_kcal*(FreshShallowOpenWater_kcal/FreshShallowOpenWater_ha),0) AS FreshShallowOpenWater_wtmean,ifnull(FreshwaterWoody_pct_kcal*(FreshwaterWoody_kcal/FreshwaterWoody_ha),0) AS FreshwaterWoody_wtmean,ifnull(MudflatSalt_pct_kcal*(MudflatSalt_kcal/MudflatSalt_ha),0) AS MudflatSalt_wtmean,ifnull(SaltMarshNonDominant_pct_kcal*(SaltMarshNonDominant_kcal/SaltMarshNonDominant_ha),0) AS SaltMarshNonDominant_wtmean'

In [290]:
con.sql(f"""
        CREATE OR REPLACE TABLE wtmean AS 
        SELECT {hucidfld}, {col_sql}
        FROM abdu_output_agg
        """)

In [291]:
col_sql = ' + '.join([f"if({i}='NaN'::FLOAT,0,{i})" for i in con.sql('describe wtmean').df()['column_name'].tolist() if i.endswith('_wtmean')])
col_sql
con.sql(f"""
        CREATE OR REPLACE TABLE wtmean AS 
        SELECT {hucidfld}, {col_sql} AS wtMean_kcal_per_ha
        FROM wtmean
        """)

In [292]:
con.sql("SELECT * FROM wtmean")

┌────────────────┬────────────────────┐
│ WATERSHED_CODE │ wtMean_kcal_per_ha │
│    varchar     │       double       │
├────────────────┼────────────────────┤
│ NS01459        │ 347482.17992075277 │
│ NS01461        │ 286192.09655392286 │
│ NS01462        │  579138.1484926699 │
│ NS01463        │  439049.9500929231 │
│ NS01464        │ 398979.37859625963 │
│ NS01465        │  285882.8713340522 │
│ NS01466        │ 239804.44593319096 │
│ NS01467        │ 243829.91287918368 │
│ NS01468        │ 333384.45697871456 │
│ NS01469        │  241041.7607135804 │
│    ·           │          ·         │
│    ·           │          ·         │
│    ·           │          ·         │
│ NS02112        │ 171848.58749000187 │
│ NS02114        │  220609.1526721803 │
│ NS02126        │ 243035.59091993506 │
│ NS02127        │ 200905.15739657317 │
│ NS02128        │ 223688.02987779403 │
│ NS02129        │ 225961.37016691142 │
│ NS02130        │ 175880.29338137293 │
│ NS02131        │ 227236.51382700502 │


In [293]:
con.execute(f"""
            CREATE OR REPLACE TABLE abdu_output_agg AS
            SELECT * EXCLUDE ({','.join(cols)}) FROM abdu_output_agg
            LEFT JOIN wtmean ON abdu_output_agg.{hucidfld}=wtmean.{hucidfld}
            """)

In [294]:
con.sql("SELECT * FROM abdu_output_agg")

┌────────────────┬──────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬──────────

# Calculate protect/restore goals

In [295]:
# Calculate protect/restore goals 
con.sql(f'''CREATE OR REPLACE TABLE abdu_output_agg AS
SELECT * EXCLUDE {hucidfld}_1,
CASE WHEN 
surpdef_lta_kcal < 0
THEN
abs(surpdef_lta_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_lta_ha,

CASE WHEN 
surpdef_80th_kcal < 0
THEN
abs(surpdef_80th_kcal/wtMean_kcal_per_ha)
ELSE 0
END
AS restoregoal_80th_ha,

CASE WHEN
wshed_ha - unavailha > 0
THEN
wshed_ha - unavailha
ELSE 0
END 
AS available_ha

FROM abdu_output_agg
''')
con.sql('''CREATE OR REPLACE TABLE abdu_output_agg AS
SELECT * EXCLUDE (restoregoal_lta_ha, restoregoal_80th_ha),
CASE WHEN 
restoregoal_lta_ha > available_ha
THEN
available_ha
ELSE restoregoal_lta_ha
END
AS restoregoal_lta_ha,

CASE WHEN 
restoregoal_80th_ha > available_ha
THEN
available_ha
ELSE restoregoal_80th_ha
END
AS restoregoal_80th_ha,

FROM abdu_output_agg
''')
#field='protectgoal_lta_ha', expression="(!nrgprot_lta_kcal!/!wtMean_kcal_per_ha!) if !nrgprot_lta_kcal! > 0 else 0"
#field='protectgoal_80th_ha', expression="(!nrgprot_80th_kcal!/!wtMean_kcal_per_ha!) if !nrgprot_80th_kcal! > 0 else 0"
con.sql('''CREATE OR REPLACE TABLE abdu_output_agg AS
SELECT *,
CASE WHEN 
nrgprot_lta_kcal > 0 
THEN
nrgprot_lta_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_lta_ha,

CASE WHEN 
nrgprot_80th_kcal > 0
THEN
nrgprot_80th_kcal/wtMean_kcal_per_ha
ELSE 0
END
AS protectgoal_80th_ha,
FROM abdu_output_agg
''')
#field='protectgoal_lta_ha', expression="!available_ha! if !protectgoal_lta_ha! > !available_ha! else !protectgoal_lta_ha!"
#field='protectgoal_80th_ha', expression="!available_ha! if !protectgoal_80th_ha! > !available_ha! else !protectgoal_80th_ha!"
con.sql('''CREATE OR REPLACE TABLE abdu_output_agg AS
SELECT * EXCLUDE (protectgoal_lta_ha, protectgoal_80th_ha),
CASE WHEN 
protectgoal_lta_ha > available_ha
THEN
available_ha
ELSE protectgoal_lta_ha
END
AS  protectgoal_lta_ha,

CASE WHEN 
protectgoal_80th_ha > available_ha
THEN
available_ha
ELSE protectgoal_80th_ha
END
AS protectgoal_80th_ha,
       
FROM abdu_output_agg
''')

In [296]:
'''
Protected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of
wetland energy

Calculations:
    Energy supply
        Total habitat energy within huc - THabNrg
        Total habitat hectares within huc - THabHA

    Energy demand
        LTA and X80 DUD by huc - TLTADUD anc X80DUD
        LTA and X80 Demand by huc - TLTADemand and X80Demand
        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj
        
    Protected lands
        Total protected hectares by huc - ProtHA

    Protected habitat hectares and energy
        Total protected hectares - ProtHabHA
        Total protected energy - ProtHabNrg

    Weighted mean and calculations based off of it
        Weighted mean kcal/ha with weight being Total habitat energy
        Energy Protection needed - NrgProtRq
        Restoration HA based off of weighted mean - RstorHA
        Protection HA based off weighted mean - RstorProtHA  

'''
#################################
#################################
#################################


'\nProtected wetlands, urban wetlands, and wetland energy all calculated by huc12.  Need to calculate total urban outside of\nwetland energy\n\nCalculations:\n    Energy supply\n        Total habitat energy within huc - THabNrg\n        Total habitat hectares within huc - THabHA\n\n    Energy demand\n        LTA and X80 DUD by huc - TLTADUD anc X80DUD\n        LTA and X80 Demand by huc - TLTADemand and X80Demand\n        LTA and X80 Population objective by huc - LTAPopObj and X80PopObj\n        \n    Protected lands\n        Total protected hectares by huc - ProtHA\n\n    Protected habitat hectares and energy\n        Total protected hectares - ProtHabHA\n        Total protected energy - ProtHabNrg\n\n    Weighted mean and calculations based off of it\n        Weighted mean kcal/ha with weight being Total habitat energy\n        Energy Protection needed - NrgProtRq\n        Restoration HA based off of weighted mean - RstorHA\n        Protection HA based off weighted mean - RstorProtHA 

# Join with hucs geometry

In [297]:
con.execute(f"""
            CREATE OR REPLACE TABLE hucs AS 
            SELECT * FROM read_parquet('{hucsurl}')
            """)
cols = con.sql("DESCRIBE hucs").df().column_name.to_list()
cols

['OBJECTID', 'WATERSHED_CODE', 'MERGE_SRC', 'geometry', 'bbox']

In [298]:
con.sql(f"SELECT count(*) AS n_hucs FROM hucs WHERE {hucidfld} IN (SELECT distinct({hucidfld}) FROM abdu_output_agg)"), con.sql(f"SELECT count(*) AS n_abdu FROM abdu_output_agg")

(┌────────┐
 │ n_hucs │
 │ int64  │
 ├────────┤
 │    306 │
 └────────┘,
 ┌────────┐
 │ n_abdu │
 │ int64  │
 ├────────┤
 │    306 │
 └────────┘)

In [299]:
cols.remove(hucidfld)
cols

['OBJECTID', 'MERGE_SRC', 'geometry', 'bbox']

In [300]:
con.execute(f"""
            CREATE OR REPLACE TABLE abdu_output_agg AS
            SELECT * EXCLUDE ({','.join(cols)}), geometry
            FROM abdu_output_agg
            LEFT JOIN hucs ON abdu_output_agg.{hucidfld}=hucs.{hucidfld}
""")
# con.execute(f"""
#             CREATE OR REPLACE TABLE abdu_output_agg AS
#             SELECT * EXCLUDE ({hucidfld}_1, geometry), ST_GeomFromWKB(geometry) AS geometry
#             FROM abdu_output_agg
#             ORDER BY {hucidfld}
# """)

In [221]:
# con.sql("SELECT * FROM abdu_output_agg WHERE contains(dgr_blk, ', ')")
# con.sql("SELECT * FROM abdu_output_agg WHERE WATERSHED_CODE='NS01759'")
con.sql("SELECT * FROM abdu_output_agg WHERE wtMean_kcal_per_ha < 1")

┌────────────────┬─────────┬────────────────────┬─────────┬─────────────────┬────────────┬──────────┬──────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬──

In [302]:
con.sql("SELECT ST_Area(geometry)*0.0001 FROM abdu_output_agg WHERE WATERSHED_CODE='NS01759'")

┌──────────────────────────────┐
│ (st_area(geometry) * 0.0001) │
│            double            │
├──────────────────────────────┤
│            24338.38514796368 │
└──────────────────────────────┘

In [113]:
con.sql("SELECT * FROM abdu_output WHERE WATERSHED_CODE='NS01759'")

┌────────────────┬────────────────────┬─────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬───────────────────┬────────────────────┬─────────────────────┬───────────────────┬────────────────────┬─────────────────────┬────────────────────┬───────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬───────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────────────────

In [204]:
(27.86344660079065+34.475336044835124+10.043528581130087)/3#/362.85589910320704
# 59.78888920990815 + 109.8987329439488 + 193.1682769493501
59.78888920990815/362.85589910320704
con.sql("SELECT sum(ST_Area(ST_GeomFromWKB(geometry))*0.0001) FROM abdu_output WHERE WATERSHED_CODE='NS01759'")

┌───────────────────────────────────────────────────┐
│ sum((st_area(st_geomfromwkb(geometry)) * 0.0001)) │
│                      double                       │
├───────────────────────────────────────────────────┤
│                                 24338.38514796368 │
└───────────────────────────────────────────────────┘

In [303]:
con.sql(f"""
        COPY (
            SELECT * EXCLUDE {hucidfld}_1
            FROM abdu_output_agg
            ORDER BY {hucidfld}
        )
        TO './output/{fname}' (FORMAT PARQUET)
        """)
fname

'abdu_cn_east_recalc.parquet'

In [241]:
con.sql(f"""
        SELECT WATERSHED_CODE, wtMean_kcal_per_ha, tothabitat_kcal 
        FROM read_parquet('output\\abdu_cn_east_wH2o.parquet')
        WHERE WATERSHED_CODE='NS01633'
        """)

┌────────────────┬────────────────────┬────────────────────┐
│ WATERSHED_CODE │ wtMean_kcal_per_ha │  tothabitat_kcal   │
│    varchar     │       double       │       double       │
├────────────────┼────────────────────┼────────────────────┤
│ NS01633        │ 184805.24556777233 │ 117466059.60591103 │
└────────────────┴────────────────────┴────────────────────┘

In [242]:
con.sql(f"""
        SELECT WATERSHED_CODE, wtMean_kcal_per_ha, tothabitat_kcal 
        FROM abdu_output_agg
        WHERE WATERSHED_CODE='NS01633'
        """)

┌────────────────┬────────────────────┬──────────────────┐
│ WATERSHED_CODE │ wtMean_kcal_per_ha │ tothabitat_kcal  │
│    varchar     │       double       │      double      │
├────────────────┼────────────────────┼──────────────────┤
│ NS01633        │ 184259.21929599997 │ 67893965.1026098 │
└────────────────┴────────────────────┴──────────────────┘

In [213]:
cols = con.sql(f"""
        SELECT {hucidfld}
        FROM read_parquet('output\\abdu_cn_east_wH2o.parquet')
        """).fetchall()
print(sorted(set([i[0] for i in cols])&set([i[0] for i in con.sql(f"SELECT {hucidfld} FROM abdu_output_agg").fetchall()])))

['NS01632', 'NS01633', 'NS01634', 'NS01635', 'NS01640', 'NS01653', 'NS01654', 'NS01655', 'NS01656', 'NS01660', 'NS01661', 'NS01662', 'NS01699', 'NS01700', 'NS01701', 'NS01702', 'NS01941', 'NS01942', 'NS01952', 'NS02017', 'NS02018', 'NS02021', 'NS02023', 'NS02024', 'NS02029']


In [239]:
new, i = con.sql(f"""
        SELECT wtMean_kcal_per_ha, DeepwaterFresh_pct_ha
        FROM abdu_output_agg
        WHERE {hucidfld}='NS01632'
        """).fetchone()
new, i

(230103.61591394994, 26.29109885037355)

In [240]:
space = '\t'
for huc_id in sorted(set([i[0] for i in cols])&set([i[0] for i in con.sql(f"SELECT {hucidfld} FROM abdu_output_agg").fetchall()])):
    new, i = con.sql(f"""
        SELECT wtMean_kcal_per_ha, DeepwaterFresh_pct_ha
        FROM abdu_output_agg
        WHERE {hucidfld}='{huc_id}'
        """).fetchone()
    old = con.sql(f"""
        SELECT wtMean_kcal_per_ha 
        FROM read_parquet('output\\abdu_cn_east_wH2o.parquet')
        WHERE {hucidfld}='{huc_id}'
        """).fetchone()[0]
    diff = abs(new-old)
    pct = diff/((new+old)*0.5)
    print(f"{huc_id}:\tnew = {new:,.2f}{space*2 if new<1 else space}old = {old:,.2f}{space*2 if old<100000 else space}diff = {pct:,.0%}{space}h2o = {i if i else 0:,.0f}%")

NS01632:	new = 230,103.62	old = 171,706.80	diff = 29%	h2o = 26%
NS01633:	new = 184,259.22	old = 184,805.25	diff = 0%	h2o = 66%
NS01634:	new = 238,144.22	old = 143,223.31	diff = 50%	h2o = 14%
NS01635:	new = 242,601.08	old = 243,333.12	diff = 0%	h2o = 6%
NS01640:	new = 239,422.66	old = 122,485.30	diff = 65%	h2o = 12%
NS01653:	new = 208,216.81	old = 154,248.82	diff = 30%	h2o = 50%
NS01654:	new = 223,293.55	old = 161,493.45	diff = 32%	h2o = 20%
NS01655:	new = 144,665.83	old = 96,884.38		diff = 40%	h2o = 82%
NS01656:	new = 170,372.25	old = 170,420.73	diff = 0%	h2o = 73%
NS01660:	new = 244,525.08	old = 105,738.03	diff = 79%	h2o = 2%
NS01661:	new = 196,902.95	old = 64,383.70		diff = 101%	h2o = 89%
NS01662:	new = 244,308.73	old = 83,222.61		diff = 98%	h2o = 3%
NS01699:	new = 540,424.96	old = 72,156.92		diff = 153%	h2o = 22%
NS01700:	new = 316,832.72	old = 175,895.64	diff = 57%	h2o = 26%
NS01701:	new = 440,035.10	old = 108,034.80	diff = 121%	h2o = 55%
NS01702:	new = 391,280.02	old = 115,573.70	

In [222]:
con.sql("SELECT * FROM abdu_output_agg WHERE wtMean_kcal_per_ha < 1")

┌────────────────┬─────────┬────────────────────┬─────────┬─────────────────┬────────────┬──────────┬──────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬────────────────┬────────────────┬─────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┬──

In [312]:
hucidfld

'WATERSHED_CODE'

In [319]:
huc_id = ['NS01483', 'NS01484', 'NS01485', 'NS01486', 'NS01487', 'NS01488', 'NS01489', 'NS01490', 'NS01491', 'NS01492', 'NS01493', 'NS01494', 'NS01495', 'NS01496', 'NS01498', 'NS01499', 'NS01500', 'NS01501', 'NS01502', 'NS01503', 'NS01504', 'NS01505', 'NS01509', 'NS01512', 'NS01630', 'NS01631', 'NS01632', 'NS01633', 'NS01635', 'NS01637', 'NS01638', 'NS01653', 'NS01654', 'NS01655', 'NS01656', 'NS01718', 'NS01741', 'NS01743', 'NS01745', 'NS01746', 'NS01759', 'NS01769', 'NS01770', 'NS01771', 'NS01772', 'NS01773', 'NS01774', 'NS01775', 'NS01776', 'NS01777', 'NS01778', 'NS01779', 'NS01780', 'NS01781', 'NS01782', 'NS01783', 'NS02009', 'NS02013', 'NS02014', 'NS02015', 'NS02016', 'NS02017', 'NS02018']
con.sql(f"""
        SELECT {hucidfld}, tothabitat_kcal, wtMean_kcal_per_ha 
        FROM read_parquet('output\\abdu_cn_east_wH2o.parquet')
        WHERE {hucidfld} IN ('{"','".join(huc_id)}')
        """)

┌────────────────┬────────────────────┬────────────────────┐
│ WATERSHED_CODE │  tothabitat_kcal   │ wtMean_kcal_per_ha │
│    varchar     │       double       │       double       │
├────────────────┼────────────────────┼────────────────────┤
│ NS01632        │ 112894589.13302337 │ 171706.80372429764 │
│ NS01633        │ 117466059.60591103 │ 184805.24556777233 │
│ NS01635        │ 176171843.00172094 │  243333.1150633344 │
│ NS01653        │ 124595746.43916556 │ 154248.82195683647 │
│ NS01654        │ 190759746.81108475 │ 161493.45417853227 │
│ NS01655        │ 155704521.76664636 │  96884.38210401376 │
│ NS01656        │ 20711215.505087916 │ 170420.72914406343 │
│ NS02017        │ 221224722.50813222 │ 119708.05405500876 │
│ NS02018        │ 1037768117.3045499 │  75956.85985620112 │
└────────────────┴────────────────────┴────────────────────┘

In [317]:
huc_id = con.sql(f"""
        SELECT distinct({hucidfld})
        FROM read_parquet('output\\abdu_cn_east_wH2o.parquet')
        WHERE {hucidfld} IN ('{"','".join(huc_id)}')
        """).fetchall()
print([i[0] for i in huc_id])

['NS01655', 'NS01632', 'NS01633', 'NS01653', 'NS02018', 'NS01656', 'NS02017', 'NS01635', 'NS01654']


# CONVERT PARQUET MODEL OUTPUT TO FEATURE CLASS

In [12]:
# arcgispro3_4clone
import geopandas as gpd
import arcpy
import os
in_parquet = r"D:\ABDUBounds\model_output\abdu_cn_east_v3.parquet"
out_fc = r"D:\ABDUBounds\model_output\abdu_model_out.gdb\abdu_cn_east_v3"

In [13]:
df = gpd.read_parquet(in_parquet)
df.set_crs('ESRI:102008', allow_override=True, inplace=True)
df.crs

<Projected CRS: ESRI:102008>
Name: North_America_Albers_Equal_Area_Conic
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. United States (USA) - Alabama; Alaska (mainland); Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming.
- bounds: (-172.54, 23.81, -47.74, 86.46)
Coordin

In [14]:
df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2382 entries, 0 to 2381
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   WATERSHED_CODE                2382 non-null   object  
 1   wshed_ha                      2382 non-null   float64 
 2   dud_lta                       2382 non-null   float64 
 3   demand_lta_kcal               2382 non-null   float64 
 4   popobj_lta                    2382 non-null   float64 
 5   dud_80th                      2382 non-null   float64 
 6   demand_80th_kcal              2382 non-null   float64 
 7   popobj_80th                   2382 non-null   float64 
 8   abdu_ltadud                   2382 non-null   float64 
 9   abdu_ltademand                2382 non-null   float64 
 10  abdu_ltapopobj                2382 non-null   float64 
 11  abdu_x80dud                   2382 non-null   float64 
 12  abdu_x80demand                2382 non-n

In [15]:
df.drop_duplicates(inplace=True)
df.shape

(2382, 86)

In [16]:
df.duplicated('WATERSHED_CODE').any()
# dbl_df = df[df.duplicated('WATERSHED_CODE', keep=False)]
# dbl_df.shape
# df2 = dbl_df.groupby('WATERSHED_CODE').agg({'tothabitat_kcal':len})
# df2.loc[:,'tothabitat_kcal_min'] = dbl_df.groupby('WATERSHED_CODE').agg({'tothabitat_kcal':min}).tothabitat_kcal
# df2.loc[:,'tothabitat_kcal_max'] = dbl_df.groupby('WATERSHED_CODE').agg({'tothabitat_kcal':max}).tothabitat_kcal
# del dbl_df
# df2
# df2[abs(df2.tothabitat_kcal_min-df2.tothabitat_kcal_max)>1]
# df.drop_duplicates(subset='WATERSHED_CODE', inplace=True)
# df.shape

False

In [17]:
### CALCULATE STANDARDIZED ABDU FIELDS

# demand_norm = 'abdu_demand_norm'
# restore_norm = 'restore_goal_norm'
# protect_norm = 'protect_goal_norm'
abdu_restore = 'abdu_norm_restoregoal_80th'
abdu_protect = 'abdu_norm_protectgoal_80th'

# Replace nulls with 0 (shouldn't pre-exist)
df.loc[:,'abdu_x80demand'] = df.abdu_x80demand.fillna(0)
df.loc[:,'restoregoal_80th_ha'] = df.restoregoal_80th_ha.fillna(0)
df.loc[:,'protectgoal_80th_ha'] = df.protectgoal_80th_ha.fillna(0)

# Get min-max values
minVal_demand, maxVal_demand = df.abdu_x80demand.min(), df.abdu_x80demand.max()
minVal_restore, maxVal_restore = df.restoregoal_80th_ha.min(), df.restoregoal_80th_ha.max()
minVal_protect, maxVal_protect = df.protectgoal_80th_ha.min(), df.protectgoal_80th_ha.max()

# normalized demand for black ducks
zval_demand = (df.abdu_x80demand-minVal_demand)/(maxVal_demand-minVal_demand)
# normalized restoration value for all dabbling ducks
zval_restore = (df.restoregoal_80th_ha-minVal_restore)/(maxVal_restore-minVal_restore)
# normalized protection value for all dabbling ducks
zval_protect = (df.protectgoal_80th_ha-minVal_protect)/(maxVal_protect-minVal_protect)
# normalized restoration goal for black ducks
df.loc[:, abdu_restore] = zval_demand * zval_restore
# normalized protection goal for black ducks
df.loc[:, abdu_protect] = zval_demand * zval_protect
# # normalized restoration goal
# df.loc[:, restore_norm] = zval_restore
# # normalized protection goal
# df.loc[:, protect_norm] = zval_protect
df[abdu_restore].describe(), df[abdu_protect].describe()

(count    0.0
 mean     NaN
 std      NaN
 min      NaN
 25%      NaN
 50%      NaN
 75%      NaN
 max      NaN
 Name: abdu_norm_restoregoal_80th, dtype: float64,
 count    2382.000000
 mean        0.001095
 std         0.021304
 min         0.000000
 25%         0.000003
 50%         0.000036
 75%         0.000228
 max         1.000000
 Name: abdu_norm_protectgoal_80th, dtype: float64)

In [18]:
df.columns

Index(['WATERSHED_CODE', 'wshed_ha', 'dud_lta', 'demand_lta_kcal',
       'popobj_lta', 'dud_80th', 'demand_80th_kcal', 'popobj_80th',
       'abdu_ltadud', 'abdu_ltademand', 'abdu_ltapopobj', 'abdu_x80dud',
       'abdu_x80demand', 'abdu_x80popobj', 'agwt_ltadud', 'agwt_ltademand',
       'agwt_ltapopobj', 'agwt_x80dud', 'agwt_x80demand', 'agwt_x80popobj',
       'amwi_ltadud', 'amwi_ltademand', 'amwi_ltapopobj', 'amwi_x80dud',
       'amwi_x80demand', 'amwi_x80popobj', 'bwte_ltadud', 'bwte_ltademand',
       'bwte_ltapopobj', 'bwte_x80dud', 'bwte_x80demand', 'bwte_x80popobj',
       'gadw_ltadud', 'gadw_ltademand', 'gadw_ltapopobj', 'gadw_x80dud',
       'gadw_x80demand', 'gadw_x80popobj', 'mall_ltadud', 'mall_ltademand',
       'mall_ltapopobj', 'mall_x80dud', 'mall_x80demand', 'mall_x80popobj',
       'nopi_ltadud', 'nopi_ltademand', 'nopi_ltapopobj', 'nopi_x80dud',
       'nopi_x80demand', 'nopi_x80popobj', 'nsho_ltadud', 'nsho_ltademand',
       'nsho_ltapopobj', 'nsho_x80dud', '

In [19]:
i = df.columns.get_loc('geometry')
i

85

In [20]:
flds = df.columns[:i].to_list()
flds.extend(df.columns[i+1:].to_list())
flds.append(df.columns[i])
flds

['WATERSHED_CODE',
 'wshed_ha',
 'dud_lta',
 'demand_lta_kcal',
 'popobj_lta',
 'dud_80th',
 'demand_80th_kcal',
 'popobj_80th',
 'abdu_ltadud',
 'abdu_ltademand',
 'abdu_ltapopobj',
 'abdu_x80dud',
 'abdu_x80demand',
 'abdu_x80popobj',
 'agwt_ltadud',
 'agwt_ltademand',
 'agwt_ltapopobj',
 'agwt_x80dud',
 'agwt_x80demand',
 'agwt_x80popobj',
 'amwi_ltadud',
 'amwi_ltademand',
 'amwi_ltapopobj',
 'amwi_x80dud',
 'amwi_x80demand',
 'amwi_x80popobj',
 'bwte_ltadud',
 'bwte_ltademand',
 'bwte_ltapopobj',
 'bwte_x80dud',
 'bwte_x80demand',
 'bwte_x80popobj',
 'gadw_ltadud',
 'gadw_ltademand',
 'gadw_ltapopobj',
 'gadw_x80dud',
 'gadw_x80demand',
 'gadw_x80popobj',
 'mall_ltadud',
 'mall_ltademand',
 'mall_ltapopobj',
 'mall_x80dud',
 'mall_x80demand',
 'mall_x80popobj',
 'nopi_ltadud',
 'nopi_ltademand',
 'nopi_ltapopobj',
 'nopi_x80dud',
 'nopi_x80demand',
 'nopi_x80popobj',
 'nsho_ltadud',
 'nsho_ltademand',
 'nsho_ltapopobj',
 'nsho_x80dud',
 'nsho_x80demand',
 'nsho_x80popobj',
 'wodu_

In [22]:
df = df[flds].copy()
df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2382 entries, 0 to 2381
Data columns (total 88 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   WATERSHED_CODE                2382 non-null   object  
 1   wshed_ha                      2382 non-null   float64 
 2   dud_lta                       2382 non-null   float64 
 3   demand_lta_kcal               2382 non-null   float64 
 4   popobj_lta                    2382 non-null   float64 
 5   dud_80th                      2382 non-null   float64 
 6   demand_80th_kcal              2382 non-null   float64 
 7   popobj_80th                   2382 non-null   float64 
 8   abdu_ltadud                   2382 non-null   float64 
 9   abdu_ltademand                2382 non-null   float64 
 10  abdu_ltapopobj                2382 non-null   float64 
 11  abdu_x80dud                   2382 non-null   float64 
 12  abdu_x80demand                2382 non-n

### remap field names and aliases to match feature layer

In [ ]:
data_dict_xls = 'D:\ABDUBounds\ABDU_Data_Dictionary_Published (1).xlsx'

import pandas as pd
data_dict = pd.read_excel(data_dict_xls, sheet_name='Black Duck Field Descriptions', header=0, index_col=0, usecols=lambda f: not f.startswith('Unnamed'))
data_dict.head()

In [24]:
sorted([f for f in flds if f in data_dict['field name'].to_list()])

['abdu_norm_protectgoal_80th',
 'abdu_norm_restoregoal_80th',
 'available_ha',
 'demand_80th_kcal',
 'demand_lta_kcal',
 'dud_80th',
 'dud_lta',
 'nrgprot_80th_kcal',
 'nrgprot_lta_kcal',
 'popobj_80th',
 'popobj_lta',
 'protected_kcal',
 'protectedhabitat_ha',
 'protectgoal_80th_ha',
 'protectgoal_lta_ha',
 'restoregoal_80th_ha',
 'restoregoal_lta_ha',
 'surpdef_80th_kcal',
 'surpdef_lta_kcal',
 'tothabitat_kcal']

In [26]:
sorted(set(flds) - set(data_dict['field name']))

['DeepwaterFresh_pct_ha',
 'FreshMarsh_pct_ha',
 'FreshShallowOpenWater_pct_ha',
 'FreshwaterWoody_pct_ha',
 'MudflatSalt_pct_ha',
 'SaltMarshNonDominant_pct_ha',
 'WATERSHED_CODE',
 'abdu_ltademand',
 'abdu_ltadud',
 'abdu_ltapopobj',
 'abdu_x80demand',
 'abdu_x80dud',
 'abdu_x80popobj',
 'agwt_ltademand',
 'agwt_ltadud',
 'agwt_ltapopobj',
 'agwt_x80demand',
 'agwt_x80dud',
 'agwt_x80popobj',
 'amwi_ltademand',
 'amwi_ltadud',
 'amwi_ltapopobj',
 'amwi_x80demand',
 'amwi_x80dud',
 'amwi_x80popobj',
 'bwte_ltademand',
 'bwte_ltadud',
 'bwte_ltapopobj',
 'bwte_x80demand',
 'bwte_x80dud',
 'bwte_x80popobj',
 'gadw_ltademand',
 'gadw_ltadud',
 'gadw_ltapopobj',
 'gadw_x80demand',
 'gadw_x80dud',
 'gadw_x80popobj',
 'geometry',
 'mall_ltademand',
 'mall_ltadud',
 'mall_ltapopobj',
 'mall_x80demand',
 'mall_x80dud',
 'mall_x80popobj',
 'nopi_ltademand',
 'nopi_ltadud',
 'nopi_ltapopobj',
 'nopi_x80demand',
 'nopi_x80dud',
 'nopi_x80popobj',
 'nsho_ltademand',
 'nsho_ltadud',
 'nsho_ltapopo

In [36]:
updt_flds = dict().fromkeys(sorted(set(flds) - set(data_dict['field name'])), "")
del updt_flds['WATERSHED_CODE']
del updt_flds['geometry']
updt_flds.keys()

dict_keys(['DeepwaterFresh_pct_ha', 'FreshMarsh_pct_ha', 'FreshShallowOpenWater_pct_ha', 'FreshwaterWoody_pct_ha', 'MudflatSalt_pct_ha', 'SaltMarshNonDominant_pct_ha', 'abdu_ltademand', 'abdu_ltadud', 'abdu_ltapopobj', 'abdu_x80demand', 'abdu_x80dud', 'abdu_x80popobj', 'agwt_ltademand', 'agwt_ltadud', 'agwt_ltapopobj', 'agwt_x80demand', 'agwt_x80dud', 'agwt_x80popobj', 'amwi_ltademand', 'amwi_ltadud', 'amwi_ltapopobj', 'amwi_x80demand', 'amwi_x80dud', 'amwi_x80popobj', 'bwte_ltademand', 'bwte_ltadud', 'bwte_ltapopobj', 'bwte_x80demand', 'bwte_x80dud', 'bwte_x80popobj', 'gadw_ltademand', 'gadw_ltadud', 'gadw_ltapopobj', 'gadw_x80demand', 'gadw_x80dud', 'gadw_x80popobj', 'mall_ltademand', 'mall_ltadud', 'mall_ltapopobj', 'mall_x80demand', 'mall_x80dud', 'mall_x80popobj', 'nopi_ltademand', 'nopi_ltadud', 'nopi_ltapopobj', 'nopi_x80demand', 'nopi_x80dud', 'nopi_x80popobj', 'nsho_ltademand', 'nsho_ltadud', 'nsho_ltapopobj', 'nsho_x80demand', 'nsho_x80dud', 'nsho_x80popobj', 'tothab_ha', 'un

In [43]:
for i in updt_flds.keys():
    if updt_flds[i] != '':
        continue
    if i.lower() in data_dict['field name'].to_list(): 
        updt_flds[i] = i.lower()
    else:
        j = [f for f in data_dict['field name'] if f.startswith(i.lower().split('_')[0]) and len(f.split('_'))>2]
        if len(j)>0:
            j = [f for f in j if i.endswith(f.split('_')[1]) and f.split('_')[2].replace('th','') in i]
            if len(j)==1:
                updt_flds[i] = j[0]
        else:
            if i.endswith('_pct_ha'):
                if 'fresh' in i.lower():
                    j = [f for f in data_dict['field name'] if f.startswith('f_') and f.split('_')[-1] in i.lower()]
                    if len(j)==1:
                        updt_flds[i] = j[0]
                if 'salt' in i.lower():
                    j = [f for f in data_dict['field name'] if f.startswith('s_') and f.split('_')[-1] in i.lower()]
                    if len(j)==1:
                        updt_flds[i] = j[0]
updt_flds['tothab_ha'] = 'tothabitat_ha'
updt_flds['urbanNrgy'] = 'urban_kcal'
updt_flds['unavailha'] = 'unavail_ha'
updt_flds          
              

{'DeepwaterFresh_pct_ha': 'f_deepwater',
 'FreshMarsh_pct_ha': 'f_marsh',
 'FreshShallowOpenWater_pct_ha': 'f_shallowopen',
 'FreshwaterWoody_pct_ha': 'f_woody',
 'MudflatSalt_pct_ha': 's_mudflat',
 'SaltMarshNonDominant_pct_ha': 's_nd_marsh',
 'abdu_ltademand': 'abdu_demand_lta_kcal',
 'abdu_ltadud': 'abdu_dud_lta',
 'abdu_ltapopobj': 'abdu_popobj_lta',
 'abdu_x80demand': 'abdu_demand_80th_kcal',
 'abdu_x80dud': 'abdu_dud_80th',
 'abdu_x80popobj': 'abdu_popobj_80th',
 'agwt_ltademand': 'agwt_demand_lta_kcal',
 'agwt_ltadud': 'agwt_dud_lta',
 'agwt_ltapopobj': 'agwt_popobj_lta',
 'agwt_x80demand': 'agwt_demand_80th_kcal',
 'agwt_x80dud': 'agwt_dud_80th',
 'agwt_x80popobj': 'agwt_popobj_80th',
 'amwi_ltademand': 'amwi_demand_lta_kcal',
 'amwi_ltadud': 'amwi_dud_lta',
 'amwi_ltapopobj': 'amwi_popobj_lta',
 'amwi_x80demand': 'amwi_demand_80th_kcal',
 'amwi_x80dud': 'amwi_dud_80th',
 'amwi_x80popobj': 'amwi_popobj_80th',
 'bwte_ltademand': 'bwte_demand_lta_kcal',
 'bwte_ltadud': 'bwte_dud_

In [44]:
k = [i for i, j in updt_flds.items() if j=='']
for i in k:
    del updt_flds[i]
k

['wshed_ha']

In [46]:
df.rename(columns=updt_flds, inplace=True)
df.info(show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2382 entries, 0 to 2381
Data columns (total 88 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   WATERSHED_CODE              2382 non-null   object  
 1   wshed_ha                    2382 non-null   float64 
 2   dud_lta                     2382 non-null   float64 
 3   demand_lta_kcal             2382 non-null   float64 
 4   popobj_lta                  2382 non-null   float64 
 5   dud_80th                    2382 non-null   float64 
 6   demand_80th_kcal            2382 non-null   float64 
 7   popobj_80th                 2382 non-null   float64 
 8   abdu_dud_lta                2382 non-null   float64 
 9   abdu_demand_lta_kcal        2382 non-null   float64 
 10  abdu_popobj_lta             2382 non-null   float64 
 11  abdu_dud_80th               2382 non-null   float64 
 12  abdu_demand_80th_kcal       2382 non-null   float64 
 13  abdu_popob

### create and populate feature class

In [12]:
sr = arcpy.SpatialReference(102008)
sr

name (Projected Coordinate System),North_America_Albers_Equal_Area_Conic
factoryCode (WKID),102008
linearUnitName (Linear Unit),Meter
name (Geographic Coordinate System),GCS_North_American_1983
factoryCode (WKID),4269
angularUnitName (Angular Unit),Degree
datumName (Datum),D_North_American_1983


In [13]:
# arcpy.Delete_management(out_fc)
arcpy.CreateFeatureclass_management(os.path.dirname(out_fc), os.path.basename(out_fc), "POLYGON", spatial_reference=sr)

<Result 'D:\\ABDUBounds\\model_output\\abdu_model_out.gdb\\abdu_cn_east_v3'>

In [49]:
flds = []
for c in df.columns:
    f = [c]
    if c in data_dict['field name'].values:
        a = data_dict[data_dict['field name']==c]['field alias'].values[0]
    else:
        a = c
    if df[c].dtype == 'object':
        f.extend(['TEXT',a,max([len(str(i)) for i in df[c].values])])
    elif df[c].dtype in ['int64', 'int32']:
        f.extend(['LONG',a])
    elif df[c].dtype in ['float64', 'float32']:
        f.extend(['FLOAT',a])
    else:
        pass
    print(f)
    if len(f)>1:
        flds.append(f)
len(flds)

['WATERSHED_CODE', 'TEXT', 'WATERSHED_CODE', 7]
['wshed_ha', 'FLOAT', 'wshed_ha']
['dud_lta', 'FLOAT', 'Long-Term Average Duck Use Days']
['demand_lta_kcal', 'FLOAT', 'Long-Term Average Energy Demand (kcal)']
['popobj_lta', 'FLOAT', 'Long-Term Average Population Objective']
['dud_80th', 'FLOAT', '80th Percentile Duck Use Days']
['demand_80th_kcal', 'FLOAT', '80th Percentile Energy Demand (kcal)']
['popobj_80th', 'FLOAT', '80th Percentile Population Objective']
['abdu_dud_lta', 'FLOAT', 'American Black Duck Long-Term Average Duck Use Days']
['abdu_demand_lta_kcal', 'FLOAT', 'American Black Duck Long-Term Average Energy Demand (kcal)']
['abdu_popobj_lta', 'FLOAT', 'American Black Duck Long-Term Average Population Objective']
['abdu_dud_80th', 'FLOAT', 'American Black Duck 80th Percentile Duck Use Days']
['abdu_demand_80th_kcal', 'FLOAT', 'American Black Duck 80th Percentile Energy Demand (kcal)']
['abdu_popobj_80th', 'FLOAT', 'American Black Duck 80th Percentile Population Objective']
['

87

In [15]:
arcpy.AddFields_management(out_fc, flds)

<Result 'D:\\ABDUBounds\\model_output\\abdu_model_out.gdb\\abdu_cn_east_v3'>

In [16]:
flds = [i[0] for i in flds]
flds.append('SHAPE@')
flds

['WATERSHED_CODE',
 'wshed_ha',
 'dud_lta',
 'demand_lta_kcal',
 'popobj_lta',
 'dud_80th',
 'demand_80th_kcal',
 'popobj_80th',
 'abdu_ltadud',
 'abdu_ltademand',
 'abdu_ltapopobj',
 'abdu_x80dud',
 'abdu_x80demand',
 'abdu_x80popobj',
 'agwt_ltadud',
 'agwt_ltademand',
 'agwt_ltapopobj',
 'agwt_x80dud',
 'agwt_x80demand',
 'agwt_x80popobj',
 'amwi_ltadud',
 'amwi_ltademand',
 'amwi_ltapopobj',
 'amwi_x80dud',
 'amwi_x80demand',
 'amwi_x80popobj',
 'bwte_ltadud',
 'bwte_ltademand',
 'bwte_ltapopobj',
 'bwte_x80dud',
 'bwte_x80demand',
 'bwte_x80popobj',
 'gadw_ltadud',
 'gadw_ltademand',
 'gadw_ltapopobj',
 'gadw_x80dud',
 'gadw_x80demand',
 'gadw_x80popobj',
 'mall_ltadud',
 'mall_ltademand',
 'mall_ltapopobj',
 'mall_x80dud',
 'mall_x80demand',
 'mall_x80popobj',
 'nopi_ltadud',
 'nopi_ltademand',
 'nopi_ltapopobj',
 'nopi_x80dud',
 'nopi_x80demand',
 'nopi_x80popobj',
 'nsho_ltadud',
 'nsho_ltademand',
 'nsho_ltapopobj',
 'nsho_x80dud',
 'nsho_x80demand',
 'nsho_x80popobj',
 'wodu_

In [17]:
with arcpy.da.InsertCursor(out_fc, flds) as cursor:
    for i in df.itertuples(index=False):
        row = list(i)
        row[-1] = arcpy.FromWKT(row[-1].wkt, sr)
        cursor.insertRow(row)

In [18]:
arcpy.GetCount_management(out_fc)

<Result '2382'>

# update field properties (add description) to feature layer

In [3]:
data_dict_xls = 'D:\ABDUBounds\ABDU_Data_Dictionary_Published (1).xlsx'

In [4]:
import pandas as pd
data_dict = pd.read_excel(data_dict_xls, sheet_name='Black Duck Field Descriptions', header=0, index_col=0, usecols=lambda f: not f.startswith('Unnamed'))
data_dict.head()


,field name,field alias,field description,field type,field length,nullable,field type2
ID,,,,,,,
0,objectid,OBJECTID,NaN,NaN,NaN,False,esriFieldTypeOID
1,huc12,HUC12 Code,The unique 12-digit hydrologic unit code for t...,locationOrPlaceName,80.0,True,esriFieldTypeString
2,huc12name,HUC12 Name,Name of the sub-watershed.,nameOrTitle,80.0,True,esriFieldTypeString
3,huc12_ha,HUC12 Hectares,Total hectares covered by the sub-watershed.,measurement,NaN,True,esriFieldTypeDouble
4,urbanha,Urban Hectares,Total urban hectares within the sub-watershed.,measurement,NaN,True,esriFieldTypeDouble


In [5]:
from arcgis.gis import GIS
gis = GIS('Pro')#'https://duinc.maps.arcgis.com/',)
gis

GIS @ https://www.arcgis.com/ version:2026.1

In [6]:
# from arcgis.features import FeatureLayerCollection
from arcgis.features import FeatureLayer
url = 'https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/ABDU_Draft_Wetland_Data_for_Atlantic_Canada_v3/FeatureServer/0'
fL = FeatureLayer(url, gis)
fL

<FeatureLayer url:"https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/ABDU_Draft_Wetland_Data_for_Atlantic_Canada_v3/FeatureServer/0">

In [7]:
flds = fL.properties['fields']
flds

[{'name': 'OBJECTID',
  'type': 'esriFieldTypeOID',
  'alias': 'OBJECTID',
  'sqlType': 'sqlTypeOther',
  'nullable': False,
  'editable': False,
  'domain': None,
  'defaultValue': None},
 {'name': 'WATERSHED_CODE',
  'type': 'esriFieldTypeString',
  'alias': 'WATERSHED_CODE',
  'sqlType': 'sqlTypeOther',
  'length': 7,
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None},
 {'name': 'wshed_ha',
  'type': 'esriFieldTypeSingle',
  'alias': 'wshed_ha',
  'sqlType': 'sqlTypeOther',
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None},
 {'name': 'dud_lta',
  'type': 'esriFieldTypeSingle',
  'alias': 'Long-Term Average Duck Use Days',
  'sqlType': 'sqlTypeOther',
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None,
  'description': "{'value': 'Long-term average daily number of duck use days that are currently supported by existing habitats in the sub-watershed.', 'fieldValueType': ''}"},
 {'name': 'demand

In [8]:
sorted([f['name'] for f in flds if f['name'] in data_dict['field name'].to_list()])

['abdu_norm_protectgoal_80th',
 'abdu_norm_restoregoal_80th',
 'available_ha',
 'demand_80th_kcal',
 'demand_lta_kcal',
 'dud_80th',
 'dud_lta',
 'nrgprot_80th_kcal',
 'nrgprot_lta_kcal',
 'popobj_80th',
 'popobj_lta',
 'protected_kcal',
 'protectedhabitat_ha',
 'protectgoal_80th_ha',
 'protectgoal_lta_ha',
 'restoregoal_80th_ha',
 'restoregoal_lta_ha',
 'surpdef_80th_kcal',
 'surpdef_lta_kcal',
 'tothabitat_kcal']

In [10]:
updt_flds = dict().fromkeys(sorted([f['name'] for f in flds if not f['name'] in data_dict['field name'].to_list()]), "")
del updt_flds['OBJECTID']
del updt_flds['Shape__Area']
del updt_flds['Shape__Length']
del updt_flds['WATERSHED_CODE']
updt_flds.keys()

dict_keys(['DeepwaterFresh_pct_ha', 'FreshMarsh_pct_ha', 'FreshShallowOpenWater_pct_ha', 'FreshwaterWoody_pct_ha', 'MudflatSalt_pct_ha', 'SaltMarshNonDominant_pct_ha', 'abdu_ltademand', 'abdu_ltadud', 'abdu_ltapopobj', 'abdu_x80demand', 'abdu_x80dud', 'abdu_x80popobj', 'agwt_ltademand', 'agwt_ltadud', 'agwt_ltapopobj', 'agwt_x80demand', 'agwt_x80dud', 'agwt_x80popobj', 'amwi_ltademand', 'amwi_ltadud', 'amwi_ltapopobj', 'amwi_x80demand', 'amwi_x80dud', 'amwi_x80popobj', 'bwte_ltademand', 'bwte_ltadud', 'bwte_ltapopobj', 'bwte_x80demand', 'bwte_x80dud', 'bwte_x80popobj', 'gadw_ltademand', 'gadw_ltadud', 'gadw_ltapopobj', 'gadw_x80demand', 'gadw_x80dud', 'gadw_x80popobj', 'mall_ltademand', 'mall_ltadud', 'mall_ltapopobj', 'mall_x80demand', 'mall_x80dud', 'mall_x80popobj', 'nopi_ltademand', 'nopi_ltadud', 'nopi_ltapopobj', 'nopi_x80demand', 'nopi_x80dud', 'nopi_x80popobj', 'nsho_ltademand', 'nsho_ltadud', 'nsho_ltapopobj', 'nsho_x80demand', 'nsho_x80dud', 'nsho_x80popobj', 'tothab_ha', 'un

In [11]:
sorted([i for i in data_dict['field name'] if not i in [f['name'] for f in flds]])

['SHAPE__Area',
 'SHAPE__Length',
 'abdu_demand_80th_kcal',
 'abdu_demand_lta_kcal',
 'abdu_dud_80th',
 'abdu_dud_lta',
 'abdu_popobj_80th',
 'abdu_popobj_lta',
 'agwt_demand_80th_kcal',
 'agwt_demand_lta_kcal',
 'agwt_dud_80th',
 'agwt_dud_lta',
 'agwt_popobj_80th',
 'agwt_popobj_lta',
 'amwi_demand_80th_kcal',
 'amwi_demand_lta_kcal',
 'amwi_dud_80th',
 'amwi_dud_lta',
 'amwi_popobj_80th',
 'amwi_popobj_lta',
 'bwte_demand_80th_kcal',
 'bwte_demand_lta_kcal',
 'bwte_dud_80th',
 'bwte_dud_lta',
 'bwte_popobj_80th',
 'bwte_popobj_lta',
 'f_aquaticbed',
 'f_deepwater',
 'f_marsh',
 'f_shallowopen',
 'f_shores',
 'f_woody',
 'fm_aquaticbed',
 'fm_marsh',
 'fm_shallowopen',
 'gadw_demand_80th_kcal',
 'gadw_demand_lta_kcal',
 'gadw_dud_80th',
 'gadw_dud_lta',
 'gadw_popobj_80th',
 'gadw_popobj_lta',
 'globalid',
 'huc12',
 'huc12_ha',
 'huc12name',
 'inpoly_fid',
 'mall_demand_80th_kcal',
 'mall_demand_lta_kcal',
 'mall_dud_80th',
 'mall_dud_lta',
 'mall_popobj_80th',
 'mall_popobj_lta',
 

In [34]:
d = dict(value=desc, fieldValueType="")
d

{'value': 'The max-min normalized 80th Percentile Habitat Protection Goal (ha) multiplied by the max-min normalized 80th Percentile American black duck energy demand.',
 'fieldValueType': ''}

In [49]:
'%s' % d

"{'value': 'The max-min normalized 80th Percentile Habitat Protection Goal (ha) multiplied by the max-min normalized 80th Percentile American black duck energy demand.', 'fieldValueType': ''}"

In [ ]:
updt_flds = []
for f in flds:
	fname = f['name']
	if fname in data_dict['field name'].to_list():
		f['alias'] = data_dict[data_dict['field name']==fname]['field alias'].values[0].strip()
		d = dict(value=data_dict[data_dict["field name"]==fname]["field description"].values[0].strip(), fieldValueType="")
		f['description'] = d
	else:
		print(fname, 'not in data dict')
	updt_flds.append(f)
updt_flds

OBJECTID not in data dict
WATERSHED_CODE not in data dict
wshed_ha not in data dict
abdu_ltadud not in data dict
abdu_ltademand not in data dict
abdu_ltapopobj not in data dict
abdu_x80dud not in data dict
abdu_x80demand not in data dict
abdu_x80popobj not in data dict
agwt_ltadud not in data dict
agwt_ltademand not in data dict
agwt_ltapopobj not in data dict
agwt_x80dud not in data dict
agwt_x80demand not in data dict
agwt_x80popobj not in data dict
amwi_ltadud not in data dict
amwi_ltademand not in data dict
amwi_ltapopobj not in data dict
amwi_x80dud not in data dict
amwi_x80demand not in data dict
amwi_x80popobj not in data dict
bwte_ltadud not in data dict
bwte_ltademand not in data dict
bwte_ltapopobj not in data dict
bwte_x80dud not in data dict
bwte_x80demand not in data dict
bwte_x80popobj not in data dict
gadw_ltadud not in data dict
gadw_ltademand not in data dict
gadw_ltapopobj not in data dict
gadw_x80dud not in data dict
gadw_x80demand not in data dict
gadw_x80popobj not

[{'name': 'OBJECTID',
  'type': 'esriFieldTypeOID',
  'alias': 'OBJECTID',
  'sqlType': 'sqlTypeOther',
  'nullable': False,
  'editable': False,
  'domain': None,
  'defaultValue': None},
 {'name': 'WATERSHED_CODE',
  'type': 'esriFieldTypeString',
  'alias': 'WATERSHED_CODE',
  'sqlType': 'sqlTypeOther',
  'length': 7,
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None},
 {'name': 'wshed_ha',
  'type': 'esriFieldTypeSingle',
  'alias': 'wshed_ha',
  'sqlType': 'sqlTypeOther',
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None},
 {'name': 'dud_lta',
  'type': 'esriFieldTypeSingle',
  'alias': 'Long-Term Average Duck Use Days',
  'sqlType': 'sqlTypeOther',
  'nullable': True,
  'editable': True,
  'domain': None,
  'defaultValue': None,
  'description': {'value': 'Long-term average daily number of duck use days that are currently supported by existing habitats in the sub-watershed.',
   'fieldValueType': ''}},
 {'name': 'deman

In [40]:
fL.manager.update_definition({'fields': updt_flds})

{'success': True}